# Final VM Deploy Notebook

Notebook nay dung khi GPU VM bi kill hoac can deploy lai Hunyuan worker. Chay tung cell tren Jupyter cua VM.

Runtime hien tai: Expo -> FastAPI backend -> local cleanup -> Hunyuan3D worker -> GLB. Khong dung Gemini/Nano Banana va khong dung TripoSR.

## 1. Check GPU and OS

In [ ]:
!nvidia-smi
!python3 --version
!free -h
!df -h | head

## 2. Clone repo and bootstrap Hunyuan worker

Cell nay cai Python venv, Hunyuan3D-2, worker FastAPI, va tao systemd service `hunyuan-worker`.

In [ ]:
%%bash
set -euo pipefail
mkdir -p ~/work
cd ~/work
if [ ! -d AI_3D_Reconstruction_Systerm/.git ]; then
  git clone https://github.com/TangDien02/AI_3D_Reconstruction_Systerm.git
fi
cd AI_3D_Reconstruction_Systerm
git pull --ff-only
bash scripts/gcp_hunyuan_worker_bootstrap.sh

## 3. Check worker health and logs

In [ ]:
!curl -s http://127.0.0.1:8010/health

In [ ]:
!sudo systemctl status hunyuan-worker --no-pager

## 4. Start Cloudflare tunnel in tmux

Sau khi cell nay chay, xem log tunnel va copy URL `https://....trycloudflare.com` vao backend `.env.local`.

In [ ]:
%%bash
set -euo pipefail
cd ~/work/AI_3D_Reconstruction_Systerm
bash deploy/scripts/start_tunnel_tmux.sh
tmux capture-pane -t tunnel -p -S -80

## 5. View worker or tunnel logs live in Jupyter

In [ ]:
import subprocess, time
from IPython.display import clear_output

SESSION = "worker"  # change to "tunnel" if needed
while True:
    clear_output(wait=True)
    print(subprocess.check_output(
        ["tmux", "capture-pane", "-t", SESSION, "-p", "-S", "-120"],
        text=True,
    ))
    time.sleep(2)

## 6. Backend env on Windows

Copy tunnel URL vao `.env.local` cua backend Windows:

```env
RECONSTRUCTION_BACKEND=hunyuan_remote
HUNYUAN_REMOTE_URL=https://YOUR_TUNNEL.trycloudflare.com
HUNYUAN_REMOTE_OUTPUT_FORMAT=glb
HUNYUAN_REMOTE_ENABLE_TEXTURE=false
HUNYUAN_REMOTE_TIMEOUT_SECONDS=1800
HUNYUAN_REMOTE_POLL_INTERVAL_SECONDS=5
IMAGE_CLEANER_BACKEND=auto
ENABLE_REMBG_CLEANER=true
CLEAN_IMAGE_MAX_SIDE=1536
CLEAN_IMAGE_PAD_RATIO=0.08
```

Restart backend Windows:

```powershell
.\deploy\scripts\start_backend_windows.ps1 -HostIp 192.168.1.5
```

## 7. Smoke tests

Tren Windows backend:

```powershell
curl.exe http://127.0.0.1:8000/health
curl.exe https://YOUR_TUNNEL.trycloudflare.com/health
```

Sau do mo Expo va reconstruct lai tu mobile.